In [1]:
import civicpy.civic as civic
import pandas as pd

import gene_ids_in_lit.functions as fn

In [2]:
evidence = civic.get_all_evidence(include_status="accepted")
molecular_profiles = civic.get_all_molecular_profiles(include_status="accepted")

In [3]:
len(evidence)

4885

In [4]:
# Build lookup by CIViC molecular profile ID
mp_by_id = {
    mp.id: mp
    for mp in molecular_profiles
}

In [5]:
rows = []

for e in evidence:
    mp = mp_by_id.get(e.molecular_profile_id)

    if mp is None:
        continue

    for x in mp.parsed_name:
        if isinstance(x, civic.Gene):
            rows.append({
                "gene_symbol": x.name,
                "entrez_id": x.entrez_id,
                "PMID": e.source.citation_id
            })

In [6]:
df = (
    pd.DataFrame(rows)
    .drop_duplicates()
    .reset_index(drop=True)
)
df

,gene_symbol,entrez_id,PMID
0,NPM1,4869,19357394
1,EGFR,1956,25668228
2,MGMT,4255,15758010
3,KRAS,3845,18946061
4,BRAF,673,21639808
...,...,...,...
2052,RHOA,387,27313181
2053,KRAS,3845,28259530
2054,EGFR,1956,29196463
2055,KIT,3815,28595259


Make merged ambiguous symbol df from alias-alias and alias-primary collisions

In [7]:
aa_colliison_df = pd.read_csv("../output/merged_aa_collision_gene_df.csv")
ap_colliison_df = pd.read_csv("../output/merged_alias_primary_collisions_df.csv")

aa_colliison_df = aa_colliison_df.rename(columns={
    "collision": "ambiguous_symbol",
})

ap_colliison_df = ap_colliison_df.rename(columns={
    "collision": "ambiguous_symbol",
})

aa_colliison_df["collision type"] = "alias-alias"
ap_colliison_df["collision type"] = "alias-primary"

collision_df = pd.concat([aa_colliison_df, ap_colliison_df], ignore_index=True)

key_cols = [
    "ambiguous_symbol",
    "NCBI_ID",
    "primary_gene_symbol"
]

collision_df = (
    collision_df
    .groupby(key_cols, as_index=False)["collision type"]
    .agg(lambda x: sorted(set(x)))
)

In [8]:
collision_df

,ambiguous_symbol,NCBI_ID,primary_gene_symbol,collision type
0,7SK,GENE ID:125050,RN7SK,[alias-primary]
1,A2M,GENE ID:3494,IGHA2,[alias-primary]
2,AAVS1,GENE ID:54776,PPP1R12C,[alias-primary]
3,ACAT1,GENE ID:6646,SOAT1,[alias-primary]
4,ACAT2,GENE ID:8435,SOAT2,[alias-primary]
...,...,...,...,...
1679,ZNF581,GENE ID:29903,CCDC106,[alias-primary]
1680,ZNF688,GENE ID:146540,ZNF785,[alias-primary]
1681,ZP1,GENE ID:57829,ZP4,[alias-primary]
1682,ZSCAN30,GENE ID:55663,ZNF446,[alias-primary]


In [9]:
collision_df = (
    collision_df.groupby(
        ["NCBI_ID", "primary_gene_symbol"],
        as_index=False
    )
    .agg({
        "ambiguous_symbol": lambda x: list(x)
    })
)

In [10]:
collision_df

,NCBI_ID,primary_gene_symbol,ambiguous_symbol
0,GENE ID:100008586,GAGE12F,[GAGE7]
1,GENE ID:100008587,RNA5-8SN5,[RNA5-8S5]
2,GENE ID:100008588,RNA18SN5,[RNA18S5]
3,GENE ID:100008589,RNA28SN5,[RNA28S5]
4,GENE ID:100009602,TRY-GTA5-4,[TRY-GTA5-2]
...,...,...,...
1667,GENE ID:9962,SLC23A2,[SLC23A1]
1668,GENE ID:9963,SLC23A1,[SLC23A2]
1669,GENE ID:9968,MED12,[OPA1]
1670,GENE ID:9988,DMTF1,[DMP1]


Remove genes from civic pmids df that are not involved in collisions

In [11]:
collision_df["entrez_id"] = (
    collision_df["NCBI_ID"]
    .str.extract(r"(\d+)", expand=False)
    .astype("Int64")
)

In [12]:
collision_df

,NCBI_ID,primary_gene_symbol,ambiguous_symbol,entrez_id
0,GENE ID:100008586,GAGE12F,[GAGE7],100008586
1,GENE ID:100008587,RNA5-8SN5,[RNA5-8S5],100008587
2,GENE ID:100008588,RNA18SN5,[RNA18S5],100008588
3,GENE ID:100008589,RNA28SN5,[RNA28S5],100008589
4,GENE ID:100009602,TRY-GTA5-4,[TRY-GTA5-2],100009602
...,...,...,...,...
1667,GENE ID:9962,SLC23A2,[SLC23A1],9962
1668,GENE ID:9963,SLC23A1,[SLC23A2],9963
1669,GENE ID:9968,MED12,[OPA1],9968
1670,GENE ID:9988,DMTF1,[DMP1],9988


make sure types are the same

In [13]:
df["entrez_id"] = df["entrez_id"].astype("Int64")

In [20]:
df_filtered = df[
    df["entrez_id"].isin(collision_df["entrez_id"])
].copy()

In [21]:
df_filtered

,gene_symbol,entrez_id,PMID
11,UGT1A1,54658,26313268
60,NRAS,4893,28275037
89,H3-3A,3020,38335473
95,EZH2,2146,33035457
96,CHEK2,11200,32343890
...,...,...,...
2013,KLF5,688,28963353
2014,PTEN,5728,24088382
2016,FGFR1,2260,34593528
2027,NRAS,4893,24950457


Query pubmed articles for ambiguous symbol

In [22]:
df_filtered = df_filtered.merge(
    collision_df[["entrez_id", "ambiguous_symbol"]],
    on="entrez_id",
    how="left"
)
df_filtered

,gene_symbol,entrez_id,PMID,ambiguous_symbol
0,UGT1A1,54658,26313268,[UGT1A]
1,NRAS,4893,28275037,[KRAS]
2,H3-3A,3020,38335473,[H3-3B]
3,EZH2,2146,33035457,[EZH1]
4,CHEK2,11200,32343890,[CDS1]
...,...,...,...,...
210,KLF5,688,28963353,[CKLF]
211,PTEN,5728,24088382,[TEP1]
212,FGFR1,2260,34593528,[FLG]
213,NRAS,4893,24950457,[KRAS]


In [ ]:
import re
import time

In [24]:
def find_aliases_in_document(
    document: dict,
    aliases: list[str],
) -> list[str]:
    """Find ambiguous gene aliases in PubTator document text."""

    found = set()

    for passage in document.get("passages", []):
        text = str(passage.get("text", ""))

        for alias in aliases:
            pattern = re.compile(
                rf"(?<!\w){re.escape(alias)}(?!\w)",
                re.IGNORECASE,
            )

            if pattern.search(text):
                found.add(alias)

    return sorted(found)

In [25]:
pmids = set(
    df_filtered["PMID"]
    .dropna()
    .astype(str)
)

In [26]:
def fetch_documents_by_pmids(
    pmids,
    batch_size=50,
):
    """Fetch PubTator documents for a collection of PMIDs."""

    sorted_pmids = sorted(
        {str(pmid) for pmid in pmids}
    )

    for batch in fn.chunked(sorted_pmids, size=batch_size):

        response = fn.get_with_retry(
            f"{fn.BASE_URL}/publications/export/biocjson",
            params={
                "pmids": ",".join(batch),
                "full": "true",
            },
            timeout=180,
        )

        result = response.json()

        if isinstance(result, list):
            documents = result

        elif isinstance(result, dict) and "PubTator3" in result:
            documents = result["PubTator3"]

        elif isinstance(result, dict) and "documents" in result:
            documents = result["documents"]

        elif isinstance(result, dict) and "id" in result:
            documents = [result]

        else:
            documents = []

        yield from documents

        time.sleep(2)

In [27]:
import importlib

importlib.reload(fn)

<module 'gene_ids_in_lit.functions' from '/Users/rsaxs014/Desktop/gene-harmony-analysis/analysis/gene_ids_in_lit/functions.py'>

In [28]:
pmids = set(
    df_filtered.loc[
        df_filtered["PMID"].astype(str).str.fullmatch(r"\d+"),
        "PMID"
    ]
    .astype(str)
)

In [29]:
documents_by_pmid = {}

for document in fetch_documents_by_pmids(pmids):

    pmid = str(
        document.get("pmid")
        or document.get("id")
        or ""
    ).strip()

    if pmid:
        documents_by_pmid[pmid] = document

In [32]:
def find_row_aliases(row):
    pmid = str(row["PMID"])

    document = documents_by_pmid.get(pmid)

    if document is None:
        return []

    return find_aliases_in_document(
        document,
        row["ambiguous_symbol"],
    )

In [33]:
def document_has_full_text(document):
    passage_types = {
        passage.get("infons", {}).get("type", "unknown")
        for passage in document.get("passages", [])
    }

    return not passage_types <= {"title", "abstract"}

In [34]:
def analyze_row(row):
    pmid = str(row["PMID"])
    document = documents_by_pmid.get(pmid)

    if document is None:
        return pd.Series({
            "full_text_available": False,
            "aliases_found": [],
        })

    return pd.Series({
        "full_text_available": document_has_full_text(document),
        "aliases_found": find_aliases_in_document(
            document,
            row["ambiguous_symbol"],
        ),
    })

In [35]:
results = df_filtered.apply(
    analyze_row,
    axis=1,
)

df_filtered = pd.concat(
    [df_filtered, results],
    axis=1,
)

In [36]:
df_filtered = df_filtered.reset_index(drop=True)

df_filtered = df_filtered[
    df_filtered["aliases_found"].apply(len) > 0
].reset_index(drop=True)

In [37]:
df_filtered

,gene_symbol,entrez_id,PMID,ambiguous_symbol,full_text_available,aliases_found
0,NRAS,4893,28275037,[KRAS],False,[KRAS]
1,NRAS,4893,25666295,[KRAS],False,[KRAS]
2,CHEK2,11200,10617473,[CDS1],False,[CDS1]
3,EZH2,2146,20081860,[EZH1],True,[EZH1]
4,PTEN,5728,17700571,[TEP1],True,[TEP1]
5,NRG1,3084,26137564,[HRG],True,[HRG]
6,NRAS,4893,24666267,[KRAS],False,[KRAS]
7,NRAS,4893,15951308,[KRAS],False,[KRAS]
8,NRAS,4893,20619739,[KRAS],False,[KRAS]
9,NRAS,4893,22650231,[KRAS],False,[KRAS]


In [40]:
dgidb_genes_df = pd.read_csv("../input/genes (4).tsv", sep="\t", comment="#")

In [41]:
dgidb_genes_df

,gene_claim_name,nomenclature,concept_id,gene_name,source_db_name,source_db_version
0,TNFRSF10B,Gene Symbol,hgnc:11905,TNFRSF10B,HGNC,20260619
1,TNFRSF10B,Gene Symbol,hgnc:11905,TNFRSF10B,Ensembl,116
2,TNFRSF10B,Gene Symbol,hgnc:11905,TNFRSF10B,NCBI,20260622
3,TNFRSF10B,Gene Symbol,hgnc:11905,TNFRSF10B,ChEMBL,37
4,SOST,Gene Symbol,hgnc:13771,SOST,HGNC,20260619
...,...,...,...,...,...,...
77492,CD2,Gene Name,hgnc:1639,CD2,NCI,14-September-2017
77493,MUC16,Gene Name,hgnc:15582,MUC16,NCI,14-September-2017
77494,CYP19A1,Gene Name,hgnc:2594,CYP19A1,NCI,14-September-2017
77495,CCNE1,Gene Symbol,hgnc:1589,CCNE1,MskImpact,May-2015


In [43]:
ambig_symbol_set = set(collision_df["ambiguous_symbol"].explode().dropna())
dgidb_genes_set = set(dgidb_genes_df["gene_name"])
ambig_symbol_set & dgidb_genes_set

{'A2M',
 'ACAT1',
 'ACAT2',
 'ACKR5',
 'ACP1',
 'ACTB',
 'ADA2',
 'ADAM18',
 'ADAM23',
 'ADCY3',
 'ADH4',
 'ADK',
 'ADRA1A',
 'AFP',
 'AGPS',
 'AGT',
 'AHRR',
 'AIP',
 'AK3',
 'AK6',
 'ALB',
 'ALG10',
 'ALG2',
 'ALPI',
 'AMD1',
 'AMN',
 'ANXA8',
 'APC',
 'APOA2',
 'APOBEC3A',
 'APPL2',
 'AQP1',
 'AQP9',
 'AR',
 'ARF1',
 'ARG1',
 'ARHGAP10',
 'ARL1',
 'ARNT',
 'ARPC4',
 'ARSB',
 'ART1',
 'ARX',
 'ASIP',
 'ASL',
 'ATF1',
 'ATL1',
 'ATP1A1',
 'ATP2B2',
 'ATR',
 'AVP',
 'AZI2',
 'B3GNT6',
 'BAP1',
 'BCAM',
 'BCR',
 'BDNF',
 'BDP1',
 'BEX1',
 'BEX2',
 'BMI1',
 'BOLA2',
 'BRAP',
 'BRCC3',
 'BRF1',
 'BRF2',
 'BRIP1',
 'BRPF1',
 'BST1',
 'BTD',
 'BTF3',
 'C1QTNF5',
 'C2',
 'C4B',
 'C4B_2',
 'C6',
 'C7',
 'C9',
 'CA11',
 'CABP1',
 'CACNA1I',
 'CAD',
 'CAMP',
 'CAPG',
 'CAPS',
 'CARF',
 'CAST',
 'CAT',
 'CBLB',
 'CBLC',
 'CCK',
 'CCL28',
 'CCN1',
 'CCNL1',
 'CCR10',
 'CCR4',
 'CD14',
 'CD1A',
 'CD34',
 'CD3G',
 'CDC14B',
 'CDH1',
 'CDH3',
 'CDKN1A',
 'CDS1',
 'CEACAM3',
 'CENPS',
 'CEP68',
 'CER

In [47]:
dgidb_genes_df[dgidb_genes_df["gene_name"].isin(ambig_symbol_set)]

,gene_claim_name,nomenclature,concept_id,gene_name,source_db_name,source_db_version
45,PTX3,Gene Symbol,hgnc:9692,PTX3,HGNC,20260619
46,PTX3,Gene Symbol,hgnc:9692,PTX3,NCBI,20260622
61,TRAC,Gene Symbol,hgnc:12029,TRAC,HGNC,20260619
62,TRAC,Gene Symbol,hgnc:12029,TRAC,Ensembl,116
63,TRAC,Gene Symbol,hgnc:12029,TRAC,NCBI,20260622
...,...,...,...,...,...,...
77428,CPS1,Gene Symbol,hgnc:2323,CPS1,FoundationOneGenes,2020-09-03
77429,HK1,Gene Name,hgnc:4922,HK1,NCI,14-September-2017
77438,BRIP1,Gene Symbol,hgnc:20473,BRIP1,FoundationOneGenes,2020-09-03
77443,SOD2,Gene Name,hgnc:11180,SOD2,NCI,14-September-2017
